# TrOCR Benchmark and Package - Kaggle (T4)

**No training here.** This notebook reads the **same locked test set** with two models and packages each one as a
ZIP for the CRMS OCR Workspace:

1. **Microsoft's base model** `microsoft/trocr-base-handwritten` (downloaded for you, so it becomes a ZIP you can keep).
2. **Your 50k fine-tuned model** (the one you add as an input).

The test set is the `test` split of your **100k synthetic dataset**. Your 100k model was scored on exactly that
split, so once these two ZIPs are uploaded, CRMS can show all three models **side by side** and on the **radar**.
(CRMS only compares models whose reports name the same dataset, the same `manifest.csv` and the same number of
test images - cell 3 and cell 7 make sure of that.)

**Setup before running**
1. Settings -> Accelerator -> **GPU T4 x2** (one T4 is used), and Settings -> Internet -> **On** (to download the base model).
2. Add Input -> your **100k dataset** (`manifest.csv` + `train/ val/ test/`). Its test split is what is scored.
3. Add Input -> your **50k model**. Either upload the model folder (`config.json`, `model.safetensors`, tokenizer files)
   as a Kaggle dataset or Model, or upload your `TrOcr-50k-broken-samples` ZIP as a dataset (Kaggle unzips it).
   Cell 4 finds both by itself.
4. In cell 3, check the two upload names (`BASE_KEY`, `FIFTY_K_KEY`) and `DATASET_LABEL`.
5. Run it with **Save Version -> Save & Run All (Commit)**. Expect about **30 minutes**.

**What you get** (Output panel)
- `<BASE_KEY>.zip` - the base model, with its test scores.
- `<FIFTY_K_KEY>.zip` - your 50k model, unchanged, with its scores on the 100k test set and the base model's scores.
- `test-errors.csv` - every test image either model read wrong, with both readings.

Each ZIP holds: `config.json`, `generation_config.json`, `model.safetensors`, `processor_config.json`,
`tokenizer.json`, `tokenizer_config.json`, `evaluation-report.json`, `base-evaluation-report.json`, `training-summary.json`.

**Read the scores fairly.** The 100k model *trained on this dataset's train split*; the base model and the 50k model
did not. So this test measures how well each one reads **100k-style pictures**, not how well it reads real ledgers.
Cell 10 also scores the texts that never appear in the 100k training split, which is the fairer part of the comparison.
Real ledger lines are civil registry data and stay on your own computer (see docs/DEVELOPER-GUIDE.md, section 5a).

In [ ]:
# 1. Imports and small helpers
import os, io, gc, json, math, time, random, shutil, hashlib, tempfile, zipfile, warnings, logging
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
warnings.filterwarnings('ignore')
logging.disable(logging.WARNING)

import numpy as np
import pandas as pd
import torch
import transformers
from PIL import Image
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

WEIGHT_FILES = ('model.safetensors', 'pytorch_model.bin')


def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as source:
        while chunk := source.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def weights_file_in(folder):
    return next((name for name in WEIGHT_FILES if os.path.isfile(os.path.join(folder, name))), None)


def md5_of(path):
    with open(path, 'rb') as source:
        return hashlib.md5(source.read()).hexdigest()


def find_datasets(root):
    """Folders under `root` that hold a dataset (a manifest.csv)."""
    found = []
    for here, dirs, files in os.walk(root):
        dirs.sort()
        if 'manifest.csv' in files:
            found.append(os.path.normpath(here))
            dirs[:] = []  # do not walk the images
    return found


def find_model_folders(root):
    """Folders under `root` that hold a TrOCR model (config.json + weights), never inside a dataset."""
    found = []
    for here, dirs, files in os.walk(root):
        dirs.sort()
        if 'manifest.csv' in files:
            dirs[:] = []
            continue
        if 'config.json' in files and any(name in files for name in WEIGHT_FILES):
            found.append(os.path.normpath(here))
    return found


print('[OK] Imports | torch', torch.__version__, '| transformers', transformers.__version__)

In [ ]:
# 2. GPU check
print('CUDA available :', torch.cuda.is_available())
n_gpu = torch.cuda.device_count()
print('GPU count      :', n_gpu)
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {p.name}  {p.total_memory/1e9:.1f} GB')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if not torch.cuda.is_available():
    raise RuntimeError('No GPU. Settings -> Accelerator -> GPU T4 x2.')

In [ ]:
# 3. CONFIG - names and settings
MODEL_NAME = 'microsoft/trocr-base-handwritten'
# The exact Hugging Face version of the base model. CRMS installs this same version in ml/models/base.
MODEL_REVISION = 'eaacaf452b06415df8f10bb6fad3a4c11e609406'
KAGGLE_INPUT = '/kaggle/input'
WORKING_DIR = '/kaggle/working'
STAGE_DIR = '/kaggle/temp' if os.path.isdir('/kaggle/temp') else os.path.join(WORKING_DIR, '_stage')  # not kept in Output

# The names you will upload the two ZIPs as in the CRMS OCR Workspace. CRMS refuses a name that already
# exists, so use a new one (or delete the old model in CRMS first).
BASE_KEY = 'TrOcr-base-handwritten'
FIFTY_K_KEY = 'TrOcr-50k-broken-100k-test'

# These two must match your 100k model's report, or CRMS will not put the models side by side.
DATASET_LABEL = 'synthetic-handwritten-samples100k'
EXPECTED_MANIFEST_SHA256 = 'da75cd4394aecbeb30c90e960c55b9d26d33090ef6d9cd1faf50d94fb8ca7832'  # '' = skip the check

DATA_ROOT = ''       # blank = the one folder with manifest.csv under /kaggle/input (your 100k dataset)
MODEL_50K_DIR = ''   # blank = the one model folder (config.json + weights) under /kaggle/input

EVAL_BATCH_SIZE = 32      # crops per generate() call
EVAL_MAX_NEW_TOKENS = 32  # CRMS reads at most 32 tokens per crop (ml/api/main.py), so the scores match the app
CHECK_DUPLICATE_IMAGES = False  # True = read all 90k pictures to ignore exact test copies in training when counting "seen" texts, as the 100k run did (slow on Kaggle: 10 minutes to hours). Scores do not depend on it
SEED = 20261102
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

print('Upload names -> base:', repr(BASE_KEY), '| 50k model:', repr(FIFTY_K_KEY))
print('Dataset label:', repr(DATASET_LABEL))

In [ ]:
# 4. Find the 100k dataset and the 50k model under /kaggle/input
datasets_found = find_datasets(KAGGLE_INPUT)
print('Datasets found (folders with manifest.csv):')
for path in datasets_found or ['(none)']:
    print('   ', path)
if DATA_ROOT:
    DATA_ROOT = os.path.normpath(DATA_ROOT)
elif len(datasets_found) == 1:
    DATA_ROOT = datasets_found[0]
elif datasets_found:
    raise RuntimeError('More than one dataset: set DATA_ROOT in cell 3 to your 100k dataset folder (one listed above).')
else:
    raise RuntimeError('No manifest.csv under /kaggle/input. Add your 100k dataset with Add Input.')
MANIFEST_CSV = os.path.join(DATA_ROOT, 'manifest.csv')
if not os.path.isfile(MANIFEST_CSV):
    raise FileNotFoundError(f'No manifest.csv in DATA_ROOT: {DATA_ROOT}')

models_found = [m for m in find_model_folders(KAGGLE_INPUT)
                if not os.path.normpath(m).startswith(DATA_ROOT + os.sep)]
print('\nModel folders found (config.json + weights):')
for path in models_found or ['(none)']:
    print('   ', path)
if MODEL_50K_DIR:
    MODEL_50K_DIR = os.path.normpath(MODEL_50K_DIR)
elif len(models_found) == 1:
    MODEL_50K_DIR = models_found[0]
elif models_found:
    raise RuntimeError('More than one model: set MODEL_50K_DIR in cell 3 to your 50k model folder (one listed above).')
else:
    raise RuntimeError('No model folder found. Add your 50k model with Add Input (a folder with config.json and model.safetensors).')
with open(os.path.join(MODEL_50K_DIR, 'config.json'), encoding='utf-8') as source:
    model_type = json.load(source).get('model_type')
if model_type != 'vision-encoder-decoder':
    raise ValueError(f'{MODEL_50K_DIR} is not a TrOCR model (model_type {model_type!r}).')

print('\nDATA_ROOT     =', DATA_ROOT)
print('MODEL_50K_DIR =', MODEL_50K_DIR)
print('  files:', sorted(os.listdir(MODEL_50K_DIR)))

In [ ]:
# 5. Metrics (dependency-free CER / WER / exact match) and a chance check
def _levenshtein(ref, hyp):
    m, n = len(ref), len(hyp)
    if m == 0:
        return n
    if n == 0:
        return m
    prev = list(range(n + 1))
    for i in range(1, m + 1):
        curr = [i] + [0] * n
        for j in range(1, n + 1):
            cost = 0 if ref[i-1] == hyp[j-1] else 1
            curr[j] = min(prev[j]+1, curr[j-1]+1, prev[j-1]+cost)
        prev = curr
    return prev[n]


def compute_metrics(references, hypotheses):
    tce = tc = twe = tw = exact = 0
    for ref, hyp in zip(references, hypotheses):
        ref, hyp = str(ref), str(hyp)
        tce += _levenshtein(list(ref), list(hyp)); tc += len(ref)
        rw, hw = ref.split(), hyp.split()
        twe += _levenshtein(rw, hw); tw += len(rw)
        if ref == hyp:
            exact += 1
    n = len(references)
    return {'cer': (tce/tc) if tc else 0.0, 'wer': (twe/tw) if tw else 0.0,
            'accuracy': (exact/n) if n else 0.0, 'exact': exact, 'total': n}


def print_metrics(m, title='EVALUATION METRICS'):
    print('='*56); print(title); print('='*56)
    print(f"  Samples evaluated : {m['total']}")
    print(f"  Exact match       : {m['exact']}/{m['total']} ({m['accuracy']*100:.2f}%)")
    print(f"  CER               : {m['cer']*100:.2f}%")
    print(f"  WER               : {m['wer']*100:.2f}%")
    if 'ms_per_image' in m:
        print(f"  Reading time      : {m['ms_per_image']:.1f} ms per image (batches of {EVAL_BATCH_SIZE}, 1 GPU)")
    print('='*56)


def mcnemar_p(fixed, broke):
    """Exact two-sided McNemar test on the images only one of two models read right.
    Below 0.05, the difference between the two models is unlikely to be chance."""
    n = fixed + broke
    if n == 0:
        return 1.0
    tail = sum(math.comb(n, i) for i in range(min(fixed, broke) + 1)) / 2 ** n
    return min(1.0, 2 * tail)


print('[OK] Metrics')

In [ ]:
# 6. Loading and reading - the way CRMS reads: greedy, full precision, at most EVAL_MAX_NEW_TOKENS tokens
def load_base_processor():
    try:
        return TrOCRProcessor.from_pretrained(MODEL_NAME, revision=MODEL_REVISION)
    except ValueError:
        # Newer transformers 5.x no longer builds this model's tokenizer from vocab.json + merges.txt.
        # Microsoft's repository has the missing tokenizer.json in an open pull request (refs/pr/11);
        # CRMS's ml/download_trocr.py uses the same file for ml/models/base.
        folder = tempfile.mkdtemp(prefix='base-processor-')  # outside /kaggle/working, so not in Output
        for name in ('preprocessor_config.json', 'tokenizer_config.json', 'special_tokens_map.json',
                     'vocab.json', 'merges.txt'):
            shutil.copy(hf_hub_download(MODEL_NAME, name, revision=MODEL_REVISION), folder)
        shutil.copy(hf_hub_download(MODEL_NAME, 'tokenizer.json', revision='refs/pr/11'), folder)
        print('Tokenizer: added tokenizer.json from refs/pr/11 (this transformers version needs it)')
        return TrOCRProcessor.from_pretrained(folder)


def prepare_for_reading(model, proc):
    """TrOCR starts decoding from the sep/eos token (id 2). Keep config and generation_config in step,
    and turn use_cache on: CRMS does the same (ml/api/main.py)."""
    tok = proc.tokenizer
    model.config.decoder_start_token_id = tok.sep_token_id
    model.config.eos_token_id = tok.sep_token_id
    model.config.pad_token_id = tok.pad_token_id
    model.config.vocab_size = model.config.decoder.vocab_size
    gen = model.generation_config
    gen.decoder_start_token_id = model.config.decoder_start_token_id
    gen.eos_token_id = model.config.eos_token_id
    gen.pad_token_id = model.config.pad_token_id
    gen.max_new_tokens = EVAL_MAX_NEW_TOKENS
    gen.max_length = None
    gen.use_cache = True
    return model


def open_rgb(path):
    with Image.open(path) as opened:
        return opened.convert('RGB')


image_pool = ThreadPoolExecutor(max_workers=8)  # opens the next pictures while the GPU reads


@torch.no_grad()
def read_images(paths, model, proc, desc):
    model.eval()
    texts, seconds = [], 0.0
    batches = [paths[i:i + EVAL_BATCH_SIZE] for i in range(0, len(paths), EVAL_BATCH_SIZE)]
    pending = image_pool.submit(lambda b: list(map(open_rgb, b)), batches[0]) if batches else None
    for i in tqdm(range(len(batches)), desc=desc, leave=False):
        images = pending.result()
        if i + 1 < len(batches):
            pending = image_pool.submit(lambda b: list(map(open_rgb, b)), batches[i + 1])
        pixel_values = proc(images=images, return_tensors='pt').pixel_values.to(device)
        torch.cuda.synchronize()
        started = time.perf_counter()
        generated = model.generate(pixel_values, max_new_tokens=EVAL_MAX_NEW_TOKENS, use_cache=True)
        torch.cuda.synchronize()
        seconds += time.perf_counter() - started
        texts.extend(text.strip() for text in proc.batch_decode(generated, skip_special_tokens=True))
    return texts, seconds


def score(frame, model, proc, title):
    """Metrics plus each image's reading."""
    predictions, seconds = read_images(frame['path'].tolist(), model, proc, title)
    result = compute_metrics(frame['label'].tolist(), predictions)
    result['ms_per_image'] = seconds / max(len(frame), 1) * 1000
    result['predictions'] = predictions
    result['correct'] = [p == r for p, r in zip(predictions, frame['label'])]
    print_metrics(result, title)
    return result


def score_subset(frame, result, mask):
    refs = [r for r, keep in zip(frame['label'], mask) if keep]
    hyps = [h for h, keep in zip(result['predictions'], mask) if keep]
    return compute_metrics(refs, hyps)


def public(result):
    """Only the numbers, for the reports."""
    return {key: result[key] for key in ('cer', 'wer', 'accuracy', 'exact', 'total', 'ms_per_image') if key in result}


print('[OK] Loading and reading helpers')

In [ ]:
# 7. The test set: the 'test' split of the 100k dataset, read the same way as in the 100k run
def kaggle_dataset_name(root):
    parts = os.path.relpath(root, KAGGLE_INPUT).replace('\\', '/').split('/')
    if parts[0] in ('.', '..'):
        return os.path.basename(os.path.normpath(root))
    if parts[0] == 'datasets' and len(parts) >= 3:
        return parts[2]
    return parts[0]


def read_manifest(root):
    frame = pd.read_csv(os.path.join(root, 'manifest.csv'), keep_default_na=False)
    missing = {'filename', 'label', 'split'} - set(frame.columns)
    if missing:
        raise ValueError(f'{root}/manifest.csv has no {sorted(missing)} column')
    frame['filename'] = frame['filename'].astype(str).str.strip()
    frame['label'] = frame['label'].astype(str).str.strip()
    frame['split'] = frame['split'].astype(str).str.strip().str.lower()
    # Rows with no text, or marked UNREADABLE, are not used (the same rule as the fine-tuning notebook).
    frame = frame[(frame['label'] != '') & (frame['label'].str.upper() != 'UNREADABLE')].copy()
    frame['path'] = [os.path.join(root, split, name) for split, name in zip(frame['split'], frame['filename'])]
    return frame.reset_index(drop=True)


DATASET_LABEL = DATASET_LABEL.strip() or kaggle_dataset_name(DATA_ROOT)
manifest = read_manifest(DATA_ROOT)
train_all = manifest[manifest['split'] == 'train'].reset_index(drop=True)
test_df = manifest[manifest['split'] == 'test'].reset_index(drop=True)
if test_df.empty:
    raise ValueError('The dataset has no usable test rows in manifest.csv.')
test_folder = set(os.listdir(os.path.join(DATA_ROOT, 'test')))
missing_files = [p for name, p in zip(test_df['filename'], test_df['path']) if name not in test_folder and not os.path.isfile(p)]
if missing_files:
    raise FileNotFoundError(f'{len(missing_files):,} test images named in manifest.csv are missing, e.g. {missing_files[:3]}')

# Which test texts also appear in the 100k TRAINING split (the 100k model trained on those; the others did not).
if CHECK_DUPLICATE_IMAGES:  # the 100k run left exact copies of test images out of its training data; so does this count
    with ThreadPoolExecutor(max_workers=8) as pool:
        test_md5 = set(tqdm(pool.map(md5_of, test_df['path']), total=len(test_df), desc='Checking test images', leave=False))
        train_md5 = list(tqdm(pool.map(md5_of, train_all['path']), total=len(train_all), desc='Checking train images', leave=False))
    train_labels = set(train_all['label'][[m not in test_md5 for m in train_md5]])
else:
    train_labels = set(train_all['label'])
test_df['label_seen'] = test_df['label'].isin(train_labels)

manifest_sha256 = sha256_file(MANIFEST_CSV)
n_seen = int(test_df['label_seen'].sum())
print(f'Test set   : {len(test_df):,} images from {DATA_ROOT}')
print(f'             {n_seen:,} test texts also appear in the 100k training split, {len(test_df) - n_seen:,} do not')
print('manifest.csv SHA-256:', manifest_sha256)
if EXPECTED_MANIFEST_SHA256:
    if manifest_sha256 == EXPECTED_MANIFEST_SHA256.lower():
        print("[OK] Same manifest as your 100k model's report: CRMS can put the models side by side.")
    else:
        print('WARNING: this manifest.csv is NOT the one your 100k model was scored on. CRMS will not compare the\n'
              '         models side by side. Add the same 100k dataset, or clear EXPECTED_MANIFEST_SHA256 in cell 3.')

In [ ]:
# 8. Microsoft's base model on the locked test set
results = {}
base_processor = load_base_processor()
base_model = prepare_for_reading(
    VisionEncoderDecoderModel.from_pretrained(MODEL_NAME, revision=MODEL_REVISION), base_processor).to(device)
print('Decoder start token id:', base_model.config.decoder_start_token_id, '(should be 2)')
results['base'] = score(test_df, base_model, base_processor, 'BASE MODEL - TEST')
# The pinned weights file scored above. CRMS checks this hash against its own ml/models/base.
base_weights_path = hf_hub_download(MODEL_NAME, 'model.safetensors', revision=MODEL_REVISION)
base_weights_sha256 = sha256_file(base_weights_path)
print('base weights SHA-256:', base_weights_sha256)

In [ ]:
# 9. Your 50k model on the same locked test set
def load_model_processor(folder):
    try:
        return TrOCRProcessor.from_pretrained(folder)
    except Exception as error:  # no usable tokenizer files in the folder: the base tokenizer has the same vocabulary
        print('Could not load the tokenizer from the 50k folder (' + type(error).__name__ + '); using the base tokenizer.')
        return base_processor


fifty_processor = load_model_processor(MODEL_50K_DIR)
fifty_model = prepare_for_reading(VisionEncoderDecoderModel.from_pretrained(MODEL_50K_DIR), fifty_processor).to(device)
print('Decoder start token id:', fifty_model.config.decoder_start_token_id, '(should be 2)')
fifty_weights_file = weights_file_in(MODEL_50K_DIR)
fifty_weights_sha256 = sha256_file(os.path.join(MODEL_50K_DIR, fifty_weights_file))
print(f'{fifty_weights_file} SHA-256:', fifty_weights_sha256)
results['fifty_k'] = score(test_df, fifty_model, fifty_processor, f'50k MODEL ({FIFTY_K_KEY}) - TEST')

In [ ]:
# 10. SIDE-BY-SIDE on the same test images (CER/WER: lower is better | exact match: higher is better)
def pct(value):
    return f'{value*100:.2f}%'


base, tuned = results['base'], results['fifty_k']
print('=' * 70)
print(f'LOCKED TEST - {DATASET_LABEL} ({len(test_df):,} images)')
print('-' * 70)
print(f"{'':16}{'Base':>14}{'50k model':>16}{'Verdict':>14}")
for key, label, lower_is_better in (('cer', 'CER', True), ('wer', 'WER', True), ('accuracy', 'Exact match', False)):
    b, f = base[key], tuned[key]
    verdict = 'same' if f == b else ('better' if (f < b) == lower_is_better else 'worse')
    print(f'{label:16}{pct(b):>14}{pct(f):>16}{verdict:>14}')

seen_mask = test_df['label_seen'].tolist()
unseen_mask = [not s for s in seen_mask]
subsets = {}
for name, mask in (('texts_in_100k_training', seen_mask), ('texts_not_in_100k_training', unseen_mask)):
    if any(mask):
        subsets[name] = {'images': sum(mask), 'base': score_subset(test_df, base, mask),
                         'fifty_k': score_subset(test_df, tuned, mask)}
for name, label in (('texts_in_100k_training', 'In 100k train'), ('texts_not_in_100k_training', 'Not in 100k train')):
    if name in subsets:
        s = subsets[name]
        print(f"{label:16}{pct(s['base']['accuracy']):>14}{pct(s['fifty_k']['accuracy']):>16}"
              f"   exact match on {s['images']:,} images")
print('-' * 70)
fixed = sum(1 for b, f in zip(base['correct'], tuned['correct']) if f and not b)
broke = sum(1 for b, f in zip(base['correct'], tuned['correct']) if b and not f)
p_value = mcnemar_p(fixed, broke)
print(f'50k vs base: fixed {fixed:,} images, broke {broke:,}. '
      f'Chance check p = {p_value:.3g} ({"unlikely to be chance" if p_value < 0.05 else "could be chance"})')
print(f"Reading time: base {base['ms_per_image']:.1f} ms, 50k {tuned['ms_per_image']:.1f} ms per image "
      f'({torch.cuda.get_device_name(0)}, batches of {EVAL_BATCH_SIZE}, full precision)')
print('=' * 70)
print('All scores are on 100k-style synthetic pictures, not real ledger scans. The 100k model trained on this '
      'dataset; these two did not.')

# Every test image either model read wrong, with both readings.
rows = [{'filename': test_df['filename'][i], 'label': test_df['label'][i], 'base': base['predictions'][i],
         'model_50k': tuned['predictions'][i], 'text_in_100k_training': bool(test_df['label_seen'][i])}
        for i in range(len(test_df)) if not (base['correct'][i] and tuned['correct'][i])]
os.makedirs(WORKING_DIR, exist_ok=True)
errors_csv = os.path.join(WORKING_DIR, 'test-errors.csv')
pd.DataFrame(rows, columns=['filename', 'label', 'base', 'model_50k', 'text_in_100k_training']).to_csv(errors_csv, index=False)
print('Errors list:', errors_csv, f'({len(rows):,} rows)')

In [ ]:
# 11. Reports and the two ZIPs, each bound to the exact weights it was measured on
# Per ZIP:  evaluation-report.json       - that model, on the locked test set
#           base-evaluation-report.json  - the base model on the same test set (CRMS installs it on the base model
#                                          only when its weights match the base model installed in CRMS)
#           training-summary.json        - what this run did (no training: a benchmark)
# CRMS checks each report against the weights in its ZIP, and compares models only on the same dataset,
# manifest.csv and test-image count.
COPY_FILES = ('config.json', 'generation_config.json', 'preprocessor_config.json', 'processor_config.json',
              'tokenizer.json', 'tokenizer_config.json', 'special_tokens_map.json', 'added_tokens.json',
              'vocab.json', 'merges.txt')
REPORT_FILES = ('evaluation-report.json', 'base-evaluation-report.json', 'training-summary.json')
evaluated_at = datetime.now(timezone.utc).isoformat()
versions = {'torch': torch.__version__, 'transformers': transformers.__version__}
shutil.rmtree(STAGE_DIR, ignore_errors=True)  # an earlier run's leftovers must never reach a ZIP
os.makedirs(STAGE_DIR)


def reading_info(metrics):
    return {'max_new_tokens': EVAL_MAX_NEW_TOKENS, 'decoding': 'greedy', 'precision': 'fp32',
            'ms_per_image': round(metrics['ms_per_image'], 1), 'device': torch.cuda.get_device_name(0)}


def crms_report(model_key, metrics, weights_name, weights_sha256):
    return {
        'schema_version': 1,
        'model_key': model_key,
        'base_model': MODEL_NAME,
        'base_model_revision': MODEL_REVISION,
        'dataset': DATASET_LABEL,
        'manifest_sha256': manifest_sha256,
        'split': 'test',
        'sample_count': int(metrics['total']),
        'metrics': {'cer': float(metrics['cer']), 'wer': float(metrics['wer']), 'exact_match': float(metrics['accuracy'])},
        'evaluated_at': evaluated_at,
        'weights_file': weights_name,
        'weights_sha256': weights_sha256,
        'reading': reading_info(metrics),
    }


def summary_of(model_key, which, weights_sha256, source):
    summary = {
        'schema': 'crms-training-summary/1',
        'created_at': evaluated_at,
        'kind': 'benchmark_only',
        'note': 'No training happened in this run. The model was read over the locked test set of the dataset below.',
        'model': {'key': model_key, 'weights_sha256': weights_sha256, 'source': source},
        'base_model': {'name': MODEL_NAME, 'revision': MODEL_REVISION, 'weights_sha256': base_weights_sha256},
        'dataset': {'label': DATASET_LABEL, 'root': DATA_ROOT, 'manifest_sha256': manifest_sha256,
                    'test_samples': len(test_df), 'test_texts_in_100k_training': int(test_df['label_seen'].sum())},
        'settings': {'EVAL_MAX_NEW_TOKENS': EVAL_MAX_NEW_TOKENS, 'EVAL_BATCH_SIZE': EVAL_BATCH_SIZE, 'SEED': SEED,
                     'decoding': 'greedy', 'precision': 'fp32'},
        'scores': {'base': public(base), 'model': public(results[which])},
        'test_subsets': {name: {'images': s['images'], 'base': s['base'], 'model': s[which]} for name, s in subsets.items()},
        'versions': versions,
    }
    if which == 'fifty_k':
        summary['model_vs_base'] = {'fixed': fixed, 'broke': broke, 'mcnemar_p': p_value}
    return summary


def write_json(folder, name, content):
    with open(os.path.join(folder, name), 'w', encoding='utf-8') as output:
        json.dump(content, output, indent=2)


def keep_only_model_files(folder):
    """Leave only what CRMS accepts in an upload."""
    for name in os.listdir(folder):
        if name not in COPY_FILES and name not in WEIGHT_FILES and name not in REPORT_FILES:
            os.remove(os.path.join(folder, name))


def verify_package(folder, expected, n=32):
    """Load the packaged folder fresh and read the first test images: it must read as the scored model did."""
    proc = TrOCRProcessor.from_pretrained(folder)
    model = prepare_for_reading(VisionEncoderDecoderModel.from_pretrained(folder), proc).to(device)
    texts, _ = read_images(test_df['path'].tolist()[:n], model, proc, 'Checking the packaged copy')
    differ = sum(a != b for a, b in zip(texts, expected[:n]))
    del model
    gc.collect()
    torch.cuda.empty_cache()
    print(f'  packaged copy re-read {len(texts)} test images: {differ} differ from the scored readings')
    if differ > max(2, n // 10):
        raise RuntimeError(f'{folder} does not read like the model that was scored. Do not upload it.')


def make_zip(folder, key):
    archive = os.path.join(WORKING_DIR, key + '.zip')
    if os.path.exists(archive):
        os.remove(archive)
    archive = shutil.make_archive(os.path.join(WORKING_DIR, key), 'zip', folder)
    with zipfile.ZipFile(archive) as packed_zip:
        packed = set(packed_zip.namelist())
    needed = {'config.json', 'generation_config.json', 'tokenizer.json', 'tokenizer_config.json', *REPORT_FILES}
    missing = sorted(n for n in needed if n not in packed)
    if not (set(WEIGHT_FILES) & packed):
        missing.append('model weights')
    if not ({'processor_config.json', 'preprocessor_config.json'} & packed):
        missing.append('processor_config.json')
    assert not missing, f'{missing} missing from {archive}'
    return archive


# ---- ZIP 1: Microsoft's base model ---------------------------------------------------------
base_dir = os.path.join(STAGE_DIR, BASE_KEY)
os.makedirs(base_dir)
base_model.save_pretrained(base_dir)       # config.json, generation_config.json (with the decoder-start fix)
base_processor.save_pretrained(base_dir)   # processor_config.json, tokenizer.json, tokenizer_config.json
assert not os.path.exists(os.path.join(base_dir, 'model.safetensors.index.json')), 'the weights were split into shards'
# The weights are the original Hugging Face file, so their hash equals the base model installed in CRMS.
shutil.copyfile(base_weights_path, os.path.join(base_dir, 'model.safetensors'))
keep_only_model_files(base_dir)
packaged_base_sha256 = sha256_file(os.path.join(base_dir, 'model.safetensors'))
assert packaged_base_sha256 == base_weights_sha256, 'packaged base weights differ from the scored ones'
write_json(base_dir, 'evaluation-report.json', crms_report(BASE_KEY, base, 'model.safetensors', base_weights_sha256))
write_json(base_dir, 'base-evaluation-report.json', crms_report('base', base, 'model.safetensors', base_weights_sha256))
write_json(base_dir, 'training-summary.json',
           summary_of(BASE_KEY, 'base', base_weights_sha256, f'{MODEL_NAME} @ {MODEL_REVISION}'))

# ---- ZIP 2: your 50k model, its own files untouched ------------------------------------------
fifty_dir = os.path.join(STAGE_DIR, FIFTY_K_KEY)
os.makedirs(fifty_dir)
for name in sorted(os.listdir(MODEL_50K_DIR)):
    if name in COPY_FILES or name in WEIGHT_FILES:
        shutil.copyfile(os.path.join(MODEL_50K_DIR, name), os.path.join(fifty_dir, name))
extra = tempfile.mkdtemp(prefix='processor-')          # fill in any tokenizer/processor file the folder lacked
fifty_processor.save_pretrained(extra)
for name in os.listdir(extra):
    if name in COPY_FILES and not os.path.exists(os.path.join(fifty_dir, name)):
        shutil.copyfile(os.path.join(extra, name), os.path.join(fifty_dir, name))
keep_only_model_files(fifty_dir)
assert sha256_file(os.path.join(fifty_dir, fifty_weights_file)) == fifty_weights_sha256, 'weights changed while copying'
write_json(fifty_dir, 'evaluation-report.json',
           crms_report(FIFTY_K_KEY, results['fifty_k'], fifty_weights_file, fifty_weights_sha256))
write_json(fifty_dir, 'base-evaluation-report.json', crms_report('base', base, 'model.safetensors', base_weights_sha256))
write_json(fifty_dir, 'training-summary.json',
           summary_of(FIFTY_K_KEY, 'fifty_k', fifty_weights_sha256, os.path.basename(MODEL_50K_DIR)))

# ---- free the GPU, check both folders read like the scored models, then zip ------------------
for name in ('base_model', 'fifty_model'):
    globals().pop(name, None)
gc.collect()
torch.cuda.empty_cache()
print('Checking the packaged folders:')
verify_package(base_dir, results['base']['predictions'])
verify_package(fifty_dir, results['fifty_k']['predictions'])
base_zip = make_zip(base_dir, BASE_KEY)
fifty_zip = make_zip(fifty_dir, FIFTY_K_KEY)
shutil.rmtree(STAGE_DIR, ignore_errors=True)  # the folders are inside the ZIPs; keep Output small
for archive, key in ((base_zip, BASE_KEY), (fifty_zip, FIFTY_K_KEY)):
    print(f'zipped -> {archive} ({os.path.getsize(archive) / 1e9:.2f} GB)   upload as {key!r}')

In [ ]:
# 12. What to download and what to do with it
print('Output files:')
for name in sorted(os.listdir(WORKING_DIR)):
    path = os.path.join(WORKING_DIR, name)
    if os.path.isfile(path):
        print(f'  {name:42} {os.path.getsize(path) / 1e6:10.1f} MB')
print()
print('Both ZIPs were scored on:', DATASET_LABEL, f'({len(test_df):,} test images, manifest {manifest_sha256[:12]}...)')
print('In the CRMS OCR Workspace, upload:')
print(f'  1. {BASE_KEY}.zip  as {BASE_KEY!r}      (the base model; CRMS already lists it as "base", this is a keepable copy)')
print(f'  2. {FIFTY_K_KEY}.zip  as {FIFTY_K_KEY!r}')
print('Your 100k model, uploaded earlier, then appears side by side with them and on the radar.')